# Input Testing

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, matplotlib.colors as mcolors
import rasterio
from rasterio.transform import Affine
from pathlib import Path
from config import TrainConfig, ModelConfig
from model_config import make_model
from inference import FloodInferenceEngine
from inference import run_inference

cfg  = TrainConfig()
mcfg = ModelConfig()
PATCH_SIZE = mcfg.patch_size

# ── Visual setup ──────────────────────────────────────────────────────────────
FLOOD_COLORS = {0: '#FFFFFF', 1: '#C6DBEF', 2: '#6BAED6', 3: '#2171B5', 4: '#08306B'}
FLOOD_LABELS = {0: 'No Flood', 1: 'Light', 2: 'Moderate', 3: 'Heavy', 4: 'Extreme'}

conf_cmap  = 'RdYlGn'
flood_colors_rgba = [(1.0, 1.0, 1.0, 0.0), mcolors.to_rgba('#C6DBEF'), mcolors.to_rgba('#6BAED6'),  mcolors.to_rgba('#2171B5'),  mcolors.to_rgba('#08306B'), ]
flood_cmap = mcolors.ListedColormap(flood_colors_rgba)
flood_norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], ncolors=5)

legend_patches = [mpatches.Patch(facecolor='none', edgecolor='gray', linewidth=1.0, linestyle='--', label='Class 0 — No Flood'),
    *[mpatches.Patch(facecolor=FLOOD_COLORS[i], edgecolor='gray', linewidth=0.5, label=f'Class {i} — {FLOOD_LABELS[i]}') for i in range(1, 5)]
]

# ── Spatial data ──────────────────────────────────────────────────────────────
data                 = np.load(cfg.output_dir / 'spatial_data.npz')
X_full               = data['spatial']
manila_patch_indices = data['manila_patch_indices']
rain_min, rain_max   = float(data['rain_min'].item()), float(data['rain_max'].item())
N_H, N_W             = int(data['n_h'].item()),        int(data['n_w'].item())

# ── GeoTIFF transform ────────────────────────────────────────────────────────
with rasterio.open('COP-30m-GMM/greater_mm_bbox_dem_cop.tif') as src:
    t = src.transform
    dem_crs = src.crs
dem_transform = Affine(t.a * PATCH_SIZE, 0, t.c, 0, t.e * PATCH_SIZE, t.f)

# ── Model ─────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = make_model(ModelConfig()).to(device)

ckpt_files = sorted(cfg.output_dir.glob('fold_*/best_model.pt'))
best_ckpt = max(ckpt_files, key=lambda f: torch.load(f, map_location='cpu', weights_only=False).get('best_monitor', -np.inf))

ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# ── Inference engine ──────────────────────────────────────────────────────────
engine = FloodInferenceEngine(
    model       = model,
    device      = device,
    rain_min    = rain_min,
    rain_max    = rain_max,
    X_spatial   = X_full,
    num_classes = cfg.num_classes,
    batch_size  = cfg.batch_size,
    manila_patch_indices = manila_patch_indices, 
)
engine.dem_transform = dem_transform
engine.dem_crs       = dem_crs

result = run_inference(
    engine       = engine,
    storm_type   = 'triangular', # Can also be 'front-loaded', 'back-loaded', 'triangular'
    depth_mm     = 77.0,
    tpeak        = 0.9,          
    output_dir   = cfg.output_dir,
    N_H          = N_H,
    N_W          = N_W,
    has_barangay = True,
    geojson_path = 'manila_barangay_geojson.geojson'
)

In [ ]:
import geopandas as gpd
import pandas as pd

_GEOJSON_PATH = 'manila_barangay_geojson.geojson'
_CITY_LABEL = {'PH1303901': 'Manila City', 'PH1307401': 'Mandaluyong', 'PH1307404': 'Marikina', 'PH1307405': 'Pasig',
    'PH1307501': 'Las Piñas', 'PH1307503': 'Parañaque', 'PH1307602': 'San Juan', 'PH1307605': 'Taguig'}

def _build_barangay_lookup(geojson_path: str) -> dict:
    gdf = gpd.read_file(geojson_path)
    gdf['adm3_pcode'] = gdf['adm3_pcode'].astype(str)

    gdf_all = gdf.dropna(subset=['psgc_code']).copy()
    gdf_all['psgc_int'] = gdf_all['psgc_code'].astype(float).astype(int)

    lookup = {}
    for _, row in gdf_all.iterrows():
        psgc = int(row['psgc_int'])
        city = _CITY_LABEL.get(str(row['adm3_pcode']), str(row['adm3_pcode']))
        
        name = row['adm4_en'] if pd.notna(row['adm4_en']) else f'Barangay (PSGC {psgc})'

        lookup[psgc] = {
            'name'      : name,
            'city'      : city,
            'adm4_pcode': row['adm4_pcode'] if pd.notna(row.get('adm4_pcode')) else '',
            'adm3_pcode': row['adm3_pcode'],
            'area_sqkm' : float(row['shape_sqkm']) if pd.notna(row.get('shape_sqkm')) else 0.0,
        }
    return lookup

BARANGAY_LOOKUP = _build_barangay_lookup(_GEOJSON_PATH)

print(f'BARANGAY_LOOKUP ready  : {len(BARANGAY_LOOKUP):,} entries')
print(f'PSGC range             : {min(BARANGAY_LOOKUP):,} – {max(BARANGAY_LOOKUP):,}')
print(f'Cities                 : {sorted(set(v["city"] for v in BARANGAY_LOOKUP.values()))}')
